In [1]:
!pip install langchain langchain-community langchain-text-splitters langchain-classic pypdf sentence-transformers faiss-cpu "transformers<5" fpdf streamlit -q

In [2]:
from fpdf import FPDF
import os

os.makedirs('documents', exist_ok=True)

pdf1 = FPDF()
for i in range(1, 7):
    pdf1.add_page()
    pdf1.set_font("Arial", size=12)
    if i == 6:
        pdf1.cell(200, 10, txt=f"Company Policy - Page {i}", ln=1, align='C')
        pdf1.multi_cell(0, 10, txt="Leave Policy: Employees are entitled to 20 days of paid leave per year. Sick leave is capped at 10 days.")
    else:
        pdf1.cell(200, 10, txt=f"Company Policy - Page {i}", ln=1, align='C')
        pdf1.multi_cell(0, 10, txt="This page intentionally left blank.")
pdf1.output("documents/Policy.pdf")

pdf2 = FPDF()
for i in range(1, 12):
    pdf2.add_page()
    pdf2.set_font("Arial", size=12)
    if i == 11:
        pdf2.cell(200, 10, txt=f"Employee Handbook - Page {i}", ln=1, align='C')
        pdf2.multi_cell(0, 10, txt="Attendance Calculation: Attendance is calculated based on badge swipes at the main entrance.")
    else:
        pdf2.cell(200, 10, txt=f"Employee Handbook - Page {i}", ln=1, align='C')
        pdf2.multi_cell(0, 10, txt="Standard handbook boilerplate text.")
pdf2.output("documents/Handbook.pdf")

print("Created Policy.pdf and Handbook.pdf in the /documents folder.")

Created Policy.pdf and Handbook.pdf in the /documents folder.


In [3]:
%%writefile app.py
import streamlit as st
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

st.set_page_config(page_title="Domain RAG Chatbot")

@st.cache_resource
def load_ai_models():
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    #LLM: Free, local, open-source model (No API keys needed)
    model_id = "google/flan-t5-base"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
    pipe = pipeline("text2text-generation", model=model, tokenizer=tokenizer, max_new_tokens=150)
    llm = HuggingFacePipeline(pipeline=pipe)

    return embeddings, llm

embeddings, llm = load_ai_models()


st.title("Domain-Specific RAG Chatbot")
st.write("Upload a PDF and ask questions. The AI will strictly answer from the document.")

with st.sidebar:
    st.header("1. Upload Documents")
    uploaded_files = st.file_uploader("Upload PDF files", type=['pdf'], accept_multiple_files=True)
    process_btn = st.button("Process Documents")

    if st.button("Clear Chat / Reset"):
        if os.path.exists("faiss_index"):
            import shutil
            shutil.rmtree("faiss_index")
        st.success("Cleared! Please upload new documents.")


if process_btn and uploaded_files:
    with st.spinner("Processing Documents..."):
        os.makedirs("temp_docs", exist_ok=True)
        all_splits = []

        for file in uploaded_files:
            file_path = os.path.join("temp_docs", file.name)
            with open(file_path, "wb") as f:
                f.write(file.getvalue())

            #Extract text
            loader = PyPDFLoader(file_path)
            docs = loader.load()

            #Chunking: 800 chars with 120 overlap
            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=800,
                chunk_overlap=120
            )
            splits = text_splitter.split_documents(docs)
            all_splits.extend(splits)

        if all_splits:
            #Store in FAISS
            vector_store = FAISS.from_documents(all_splits, embeddings)
            vector_store.save_local("faiss_index")
            st.success("Documents processed and vector database created!")
        else:
            st.error("No text found in PDFs.")


if os.path.exists("faiss_index"):
    vector_store = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})

    prompt_template = """You are a document question-answering assistant.
Answer only from the supplied context. If the answer is not available, say:
"I could not find this information in the uploaded documents." Do not invent facts.
Mention the source document and page number when available.

Context: {context}
Question: {question}
Answer:"""

    PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True,
        chain_type_kwargs={"prompt": PROMPT}
    )

    st.divider()
    st.header("2. Ask Questions")
    user_question = st.text_input("Ask a question about your documents:")

    if user_question:
        with st.spinner("Searching for answer..."):
            result = qa_chain.invoke({"query": user_question})
            answer = result["result"]
            source_docs = result["source_documents"]

            st.write("**Answer:**")
            st.info(answer)

            st.write("**Sources used:**")
            for i, doc in enumerate(source_docs):
                source_file = os.path.basename(doc.metadata.get('source', 'Unknown'))
                page_num = doc.metadata.get('page', 0) + 1
                st.caption(f"Source {i+1}: {source_file}, Page {page_num}")

Overwriting app.py


In [4]:
!wget -q -c -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!streamlit run app.py &> /dev/null &
!./cloudflared-linux-amd64 tunnel --url http://localhost:8501

2026-08-15T01:56:03Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-15T01:56:03Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-15T01:56:06Z INF +--------------------------------------------------------------------------------------------+
2026-08-15T01:56:06Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-08-15T01:56:06Z INF |  https://comparisons-municipality-week-commons.tryclou